# Post-processing: Example 01 — Global 3D (S362ANI)

This notebook visualizes the output of the 3D S362ANI global simulation,
and optionally compares it with the 1D PREM results from example `00_global_1D`.

**Before running this notebook:**
- Run the 3D simulation from this folder: `mpirun -np 4 axisem3d --input input --output output`
- Output should appear in `output/` inside this folder
- *(Optional)* Also run example `00_global_1D` to enable 1D vs 3D comparison plots

**What this notebook does:**
1. Reads the event location from the input file
2. Plots a map of the event and GSN stations
3. Reads and plots 3D seismograms at a selected GSN station (with optional 1D comparison)
4. Processes seismograms with ObsPy (filter, resample, save to SAC)
5. Reads USArray data and makes a wavefield animation

# Import libraries

In [ ]:
import os
import yaml
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import matplotlib
matplotlib.rcParams.update({'font.size': 8})

from obspy.core import Stream, Trace, UTCDateTime, Stats
from obspy.io.sac import SACTrace

# Path to the output of this (3D) simulation
simu_dir_3D = './output'

# Path to the 1D simulation output for comparison.
# Run example 00_global_1D first, then this will be loaded automatically.
# Set to None to disable comparison.
simu_dir_1D = '../00_global_1D/output'

compare_1D = simu_dir_1D is not None and os.path.isdir(simu_dir_1D)
if compare_1D:
    print('1D results found — will plot 1D vs 3D comparison.')
else:
    print('1D results not found — plotting 3D only. Run example 00_global_1D to enable comparison.')

# Event

In [ ]:
# read event location
with open('./input/inparam.source.yaml', 'r') as file:
    source_yaml = yaml.load(file, Loader=yaml.FullLoader)
loc_leaf = source_yaml['list_of_sources'][0]['VIRGINIA_201108231751A']['location']
event_latlon = loc_leaf['latitude_longitude']
event_depth = loc_leaf['depth']

# GSN

### Station info and map

In [ ]:
# read station info
info_GSN = np.loadtxt('./input/GSN.txt', dtype=str, skiprows=3)

####################################
# draw a map of event and stations #
####################################

plt.figure(dpi=150)
map = Basemap(projection='cyl', resolution='l', lon_0=0)
map.drawcoastlines(linewidth=0.25)
map.fillcontinents(color='ivory',lake_color='lightblue')
map.drawmapboundary(fill_color='lightblue', linewidth=0)
map.scatter(event_latlon[1], event_latlon[0], latlon=True,
            s=150, c='r', marker='*', lw=0, zorder=100)
map.scatter(info_GSN[:, 3].astype(float), info_GSN[:, 2].astype(float), latlon=True,
            s=30, c='b', marker=7, lw=0, zorder=10)
plt.show()

### Read and plot seismograms

Plots the 3D (S362ANI) seismogram at a selected GSN station.
If example `00_global_1D` has been run, the 1D (PREM) result is overlaid as a dashed line for comparison.

In [ ]:
# specify a station key (network.name)
station_key = 'IU.ANMO'

# read 3D results
gsn_dir_3D = simu_dir_3D + '/stations/global_seismic_network_GSN'
time = np.loadtxt(gsn_dir_3D + '/data_time.ascii')
disp3D = np.loadtxt(gsn_dir_3D + '/%s.ascii' % station_key)

# read 1D results if available
if compare_1D:
    gsn_dir_1D = simu_dir_1D + '/stations/global_seismic_network_GSN'
    disp1D = np.loadtxt(gsn_dir_1D + '/%s.ascii' % station_key)

# plot
fig, ax = plt.subplots(3, sharex=True, dpi=150)
for ich, ch in enumerate('RTZ'):
    ax[ich].plot(time, disp3D[:, ich] * 1e6, lw=1, label='3D S362ANI')
    if compare_1D:
        ax[ich].plot(time, disp1D[:, ich] * 1e6, lw=1, label='1D PREM', linestyle='--')
    ax[ich].text(.95, .2, 'channel = ' + ch, transform=ax[ich].transAxes, ha='right', va='top')
ax[1].set_ylabel('Amplitude (mm)')
ax[0].set_xlim(time[0], time[-1])
ax[0].set_title(station_key)
plt.xlabel('Time after source origin (s)')
plt.legend()
plt.show()

### Processing using obspy

In [ ]:
# trace header
stats = Stats()
stats.starttime = UTCDateTime(time[0])
stats.delta = UTCDateTime(time[1] - time[0])
stats.npts = len(time)

# stream (3D)
stream = Stream()
for ich, ch in enumerate('RTZ'):
    stats.channel = ch
    stream.append(Trace(disp3D[:, ich], header=stats))

# process (filter, resample, slice, ...)
stream.filter('lowpass', freq=1/50)
stream.resample(1.)
stream = stream.slice(UTCDateTime(0.), UTCDateTime(1800.))

# print & plot
print(stream)
stream.plot()
plt.show()

### Save to SAC after down-sampling

In [ ]:
# create dir
os.makedirs(gsn_dir_3D + '/sac', exist_ok=True)

# sac header
sac_header = {}
sac_header['evla'] = event_latlon[0]
sac_header['evlo'] = event_latlon[1]
sac_header['evdp'] = float(event_depth) / 1e3

# loop over stations
print('Saving to SAC...')
for ist, st in enumerate(info_GSN):
    print('%d / %d' % (ist + 1, len(info_GSN)), end='\r')
    sac_header['kstnm'] = st[0]
    sac_header['knetwk'] = st[1]
    sac_header['stla'] = float(st[2])
    sac_header['stlo'] = float(st[3])
    sac_header['stdp'] = float(st[5])
    disp = np.loadtxt(gsn_dir_3D + '/%s.%s.ascii' % (st[1], st[0]))
    for ich, ch in enumerate('RTZ'):
        sac_header['kcmpnm'] = ch
        stats.sac = sac_header
        tr = Trace(data=disp[:, ich], header=stats)
        tr.resample(1.)
        tr = tr.slice(UTCDateTime(0.), UTCDateTime(1800.))
        sac = SACTrace.from_obspy_trace(tr)
        sac.write(gsn_dir_3D + '/sac/%s.%s.%s.sac' % (st[1], st[0], ch))
print('Done with %d stations.' % len(info_GSN))

# USArray

### Station info

In [ ]:
# read station locations
info_US_TA = np.loadtxt('./input/US_TA.txt', dtype=str, skiprows=3)

# dict: station key -> [lat, lon]
nstation = len(info_US_TA)
stlatlon_dict = {}
for ist in np.arange(nstation):
    key = info_US_TA[ist, 1] + '.' + info_US_TA[ist, 0]
    stlatlon_dict[key] = np.array([float(info_US_TA[ist, 2]), float(info_US_TA[ist, 3])])

### Rank-to-station map

In [ ]:
# data dir
us_ta_dir = simu_dir_3D + '/stations/USArray_transportable'

# read rank-station info
rank_station_info = np.loadtxt(us_ta_dir + '/rank_station.info', dtype=str, skiprows=1)

# dict: mpi-rank -> [station keys]
rank_station_dict = {}
stlatlon_in_data_order = []

for item in rank_station_info:
    rank = item[0]
    stkey = item[1]
    if rank not in rank_station_dict.keys():
        rank_station_dict[rank] = []
    rank_station_dict[rank].append(stkey)
    stlatlon_in_data_order.append(stlatlon_dict[stkey])

stlatlon_in_data_order = np.array(stlatlon_in_data_order)

# read time
time = np.loadtxt(us_ta_dir + '/data_time.ascii')
ntime = len(time)

### Animations on array

In [ ]:
# choose a channel to animate
# U3   -- vertical displacement
# E_I1 -- trace of strain
# R3   -- vertical rotation
channel = 'R3'

if channel == 'U3':
    norm = 1e-6
elif channel == 'E_I1':
    norm = 1e-11
elif channel == 'R3':
    norm = 1e-11
else:
    assert False, "Invalid channel."

data = np.ndarray((ntime, nstation))

pos = 0
for rank in rank_station_dict.keys():
    data_on_rank = np.loadtxt('%s/dir_rank%s/%s.ascii' % (us_ta_dir, rank, channel))
    data[:, pos:pos + len(rank_station_dict[rank])] = data_on_rank
    pos += len(rank_station_dict[rank])

In [ ]:
#############################
###### plot a snapshot ######
#############################

# specify a time step (0~384)
tstep = 100

plt.figure(dpi=150)
plt.gca().axis('off')
plt.scatter(stlatlon_in_data_order[:, 1], stlatlon_in_data_order[:, 0], s=1,
            c=data[tstep, :], vmin=-norm, vmax=norm, cmap='coolwarm')
plt.text(0, 0, 'Time = %.1f s' % (time[tstep]), transform=plt.gca().transAxes)
plt.colorbar(orientation='vertical', shrink=.5, label=channel)
plt.gca().set_aspect(1.3)
plt.show()

In [ ]:
############################
###### make animation ######
############################

os.makedirs(us_ta_dir + '/animation', exist_ok=True)

print('Making snapshots...')
for tstep in np.arange(len(time)):
    print('%d / %d' % (tstep + 1, len(time)), end='\r')
    plt.figure(dpi=150)
    plt.gca().axis('off')
    plt.scatter(stlatlon_in_data_order[:, 1], stlatlon_in_data_order[:, 0], s=1,
                c=data[tstep, :], vmin=-norm, vmax=norm, cmap='coolwarm')
    plt.text(0, 1, 'Time = %.1f s' % (time[tstep]), transform=plt.gca().transAxes)
    plt.colorbar(orientation='vertical', shrink=.5, label=channel)
    plt.gca().set_aspect(1.3)
    plt.savefig(us_ta_dir + '/animation/%s.%04d.png' % (channel, tstep))
    plt.close()

print('Creating video using ffmpeg...')
os.system("ffmpeg -y -i %s/animation/%s.%%04d.png %s/animation/%s.mp4" %
          (us_ta_dir, channel, us_ta_dir, channel))
os.system('rm ' + us_ta_dir + '/animation/%s.*.png' % (channel,))
print('Done.')

In [ ]:
# play animation
from IPython.display import Video
Video("%s/animation/%s.mp4" % (us_ta_dir, channel))